# GuindiMa — les nouvelles réponses de conversation, voix Oolel native (sur Brev)

Quatre réponses de conversation récemment validées, à installer sans attendre la
prochaine régénération complète, plus un essai d'écoute « Tata » : trois prises
de « Jëlal Tataa ñett. » / « Jëlal Tatà ñett. » / « Jëlal Tata ñett. » pour
comparer la lecture du nom de la compagnie par la voix.

Mêmes réglages que partout : graine 1234, 3 essais par audio, Whisper wolof
garde le meilleur.

**Machine :** GPU NVIDIA (T4 ou L4) chez Brev — pas de Google Drive : tout est
écrit dans `sortie_voix/`, à côté du notebook. Voir `notebooks/BREV_LIRE_MOI.md`.

**Jeton 🔑 :** la cellule Whisper lit le jeton dans la **variable
d'environnement `HF_TOKEN`** de la machine — pas de saisie cachée : sans
interface, il n'y a pas de clavier au premier plan. Jeton absent → Whisper
wolof public de contrôle directement (le dépôt privé ne serait de toute façon
pas accessible).

**Lancer :** la première cellule installe pip (ensurepip) et les paquets, puis
**Kernel -> Restart Kernel** et Run All — voir la note sous le pip : sans
redémarrage, numpy plante avec « Numpy is not available ». Sans interface :

    jupyter nbconvert --to notebook --execute --ExecutePreprocessor.timeout=-1 voix_ajouts_conversation_brev.ipynb

La reprise saute ce qui est déjà fait dans `sortie_voix/audios_conversation/`.

À la fin, zip `sortie_voix/ajouts_conversation.zip` (panneau de gauche de
Jupyter). Sur ton PC, seules les 4 réponses s'installent — les essais « Tata »
restent à écouter :

    python scripts/installer_voix_oolel.py --conversation sortie_voix/ajouts_conversation.zip


In [ ]:
# Brev : le Jupyter de base démarre sans pip, sans torch, sans numpy.
# ensurepip installe pip dans le noyau, sans option (il refuse les
# options de pip) ; le pip du noyau installe ensuite la pile
# torch/numpy + les paquets des versions Colab, à l'identique.
import sys

!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install torch torchaudio torchcodec huggingface_hub numpy omegaconf einops scipy safetensors resemble-perth pyloudnorm 'librosa>=0.10.2' transformers==4.46.3 diffusers==0.29.0 conformer==0.3.2 s3tokenizer num2words soundfile


> **Après cette cellule : Kernel -> Restart Kernel, puis Run All.**
>
> Pourquoi : une fois torch chargé, numpy du même noyau plante avec
> « Numpy is not available ». Le redémarrage remet les deux d'accord, et la
> reprise saute les audios déjà écrits dans `sortie_voix/` : rien n'est refait
> pour rien.


In [ ]:
# Tout atterrit ici, à côté du notebook : audios, rapport, voix de référence.
import os, glob, shutil

BASE = './sortie_voix'
SORTIE = f'{BASE}/audios_conversation'
RAPPORT = f'{BASE}/rapport_ajouts_conversation.csv'
REF = f'{BASE}/8_1_c.wav'
os.makedirs(SORTIE, exist_ok=True)
if not os.path.exists(REF):
    from huggingface_hub import hf_hub_download
    telecharge = hf_hub_download(
        repo_id='soynade-research/Oolel-Voices-Demo',
        repo_type='space',
        filename='8_1_c.wav',
    )
    shutil.copy(telecharge, REF)
print('voix de référence :', REF)
print('déjà faits :', len(glob.glob(f'{SORTIE}/*.wav')), '/ 7')


In [ ]:
PLAN = {'modele': 'soynade-research/Oolel-Voices',
 'voix_reference': '8_1_c.wav (Space soynade-research/Oolel-Voices-Demo)',
 'graine': 1234,
 'phrases': [{'fichier': 'conversation_ci_lo_nekk.wav',
              'phrase': 'Maa ngi ci jàmm, alxamdulilaay. Yaw nag ?',
              'type': 'conversation'},
             {'fichier': 'conversation_liggeey.wav',
              'phrase': 'Sama liggéey mooy won nit ñi biis bi ñuy jël ci Ndakaaru.',
              'type': 'conversation'},
             {'fichier': 'conversation_dekk.wav',
              'phrase': 'Ci Ndakaaru laa dëkk, ci biir sa telefon.',
              'type': 'conversation'},
             {'fichier': 'conversation_wolof.wav',
              'phrase': 'Waaw, dégg naa wolof. Wax ma fan nga bëgga dem.',
              'type': 'conversation'},
             {'fichier': 'test_tata_tataa.wav', 'phrase': 'Jëlal Tataa ñett.', 'type': 'test_tata'},
             {'fichier': 'test_tata_tata_accent.wav', 'phrase': 'Jëlal Tatà ñett.', 'type': 'test_tata'},
             {'fichier': 'test_tata_tata.wav', 'phrase': 'Jëlal Tata ñett.', 'type': 'test_tata'}]}
PHRASES = PLAN['phrases']
GRAINE = PLAN['graine']
print(len(PHRASES), 'phrases')


In [ ]:
# HELPERS_VERIFICATION_VOIX
# Ces fonctions sont pures (ni torch ni numpy) : les tests locaux exécutent
# directement cette cellule pour vérifier le choix du meilleur candidat.
import re
import unicodedata


SEUIL_SIMILARITE = 0.5
VITESSE_CAR_PAR_SEC = 12.0
TOLERANCE_MIN = 0.8
TOLERANCE_RATIO = 0.6
DUREE_MIN = 0.45


def normaliser_texte(texte):
    """Minuscules, sans accents ni ponctuation, pour comparer Whisper au plan."""
    texte = (texte or "").lower()
    texte = texte.translate(str.maketrans({"ŋ": "n", "ë": "e", "ñ": "n"}))
    texte = unicodedata.normalize("NFKD", texte)
    texte = "".join(c for c in texte if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]+", " ", texte).strip()


def distance_levenshtein(a, b):
    """Distance de caractères classique, en O(min(len(a), len(b))) mémoire."""
    if len(a) < len(b):
        a, b = b, a
    precedente = list(range(len(b) + 1))
    for i, caractere_a in enumerate(a, 1):
        courante = [i]
        for j, caractere_b in enumerate(b, 1):
            cout = 0 if caractere_a == caractere_b else 1
            courante.append(min(
                courante[-1] + 1,
                precedente[j] + 1,
                precedente[j - 1] + cout,
            ))
        precedente = courante
    return precedente[-1]


def similarite(phrase, transcription):
    """Score de 0 à 1 : 1 = transcription identique après normalisation."""
    a = normaliser_texte(phrase)
    b = normaliser_texte(transcription)
    if not a and not b:
        return 1.0
    return 1.0 - distance_levenshtein(a, b) / max(len(a), len(b))


def bornes_duree(phrase):
    """Bornes basses et hautes attendues pour une phrase, en secondes."""
    longueur = len(normaliser_texte(phrase))
    attendu = max(DUREE_MIN, longueur / VITESSE_CAR_PAR_SEC)
    tolerance = max(TOLERANCE_MIN, attendu * TOLERANCE_RATIO)
    return max(DUREE_MIN, attendu - tolerance), attendu + tolerance


def controler_duree(duree_s, phrase):
    """Renvoie (ok, raison) ; la borne haute attrape le son après la fin."""
    mini, maxi = bornes_duree(phrase)
    if duree_s < mini:
        return False, "trop court"
    if duree_s > maxi:
        return False, "trop long / son après la fin attendue"
    return True, ""


def est_douteux(score, duree_ok):
    """Douteux si Whisper doute ou si la durée est hors bornes."""
    return score < SEUIL_SIMILARITE or not duree_ok


def choisir_meilleur_candidat(candidats):
    """Indice du meilleur candidat : similarité, puis durée valide.

    Un candidat trop court ou trop long est pénalisé de moitié : mieux vaut
    une phrase un peu moins bien reconnue qu'un WAV coupé ou prolongé.
    """
    if not candidats:
        raise ValueError("aucun candidat")

    def cle(indice):
        candidat = candidats[indice]
        ajuste = candidat["score"] if candidat.get("duree_ok", True) else candidat["score"] * 0.5
        return ajuste, candidat["score"], -indice

    return max(range(len(candidats)), key=cle)


In [ ]:
import torch
device = 'cuda:0'
assert torch.cuda.is_available(), 'Pas de GPU visible : choisis une machine T4 ou L4 (voir notebooks/BREV_LIRE_MOI.md)'
# Whisper de contrôle : notre fine-tuning Dakar d'abord, repli public ensuite.
import os

import numpy as np
from transformers import WhisperForConditionalGeneration, WhisperProcessor

WHISPER_PRINCIPAL = "PythonMindset/whisper-small-wolof-dakar-essai3"
WHISPER_SECOURS = "M9and2M/whisper-small-wolof"

# Jeton uniquement dans la variable d'environnement HF_TOKEN : pas de saisie
# cachée ici, il n'y a pas de clavier au premier plan (surtout sans interface).
HF_TOKEN = (os.environ.get("HF_TOKEN") or "").strip() or None

if HF_TOKEN is None:
    # Sans jeton, le dépôt privé est inaccessible de toute façon : direction
    # directement le Whisper wolof public de contrôle.
    print("Jeton absent : Whisper wolof public de contrôle directement.")
    WHISPER_PRINCIPAL = WHISPER_SECOURS


def charger_whisper():
    for depot, jeton in ((WHISPER_PRINCIPAL, HF_TOKEN), (WHISPER_SECOURS, None)):
        try:
            try:
                processeur = WhisperProcessor.from_pretrained(depot, token=jeton)
            except Exception:
                # Sans preprocessor_config.json dans le dépôt (c'est le cas de
                # PythonMindset/whisper-small-wolof-dakar-essai3), on lit celui du Whisper-small d'origine, public.
                # Les poids restent ceux du dépôt demandé, seule la recette de
                # spectrogramme vient d'ailleurs — elle est la même.
                processeur = WhisperProcessor.from_pretrained(
                    "openai/whisper-small", token=jeton
                )
            modele = WhisperForConditionalGeneration.from_pretrained(depot, token=jeton).to(device)
            modele.eval()
            print("Whisper de contrôle :", depot)
            return processeur, modele, depot
        except Exception as exc:
            print(f"[Whisper] {depot} inaccessible ({type(exc).__name__} : {exc})")
    raise RuntimeError(f"ni {WHISPER_PRINCIPAL} ni {WHISPER_SECOURS} ne se chargent")


asr_proc, asr_model, asr_depot = charger_whisper()


def reechantillonner_16k(audio, sr):
    """Whisper attend 16 kHz mono ; interpolation linéaire sans dépendance."""
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    if sr == 16000 or audio.size == 0:
        return audio
    n = int(round(len(audio) * 16000 / sr))
    if n <= 1:
        return audio[:n]
    x_ancien = np.linspace(0.0, 1.0, len(audio), endpoint=False, dtype=np.float64)
    x_nouveau = np.linspace(0.0, 1.0, n, endpoint=False, dtype=np.float64)
    return np.interp(x_nouveau, x_ancien, audio).astype(np.float32)


def transcrire(audio, sr):
    """Transcrit un WAV déjà généré (pas de language : « wo » fait planter Whisper)."""
    audio_16k = reechantillonner_16k(audio, sr)
    entree = asr_proc(audio_16k, sampling_rate=16000, return_tensors="pt")
    with torch.inference_mode():
        ids = asr_model.generate(entree.input_features.to(device), max_new_tokens=96)
    return asr_proc.batch_decode(ids, skip_special_tokens=True)[0].strip()


In [ ]:
# Oolel-Voices
import torchaudio
from huggingface_hub import snapshot_download
from transformers import AutoModel
oolel = AutoModel.from_pretrained(snapshot_download('soynade-research/Oolel-Voices'), trust_remote_code=True)
SR = oolel.sr
print('Oolel chargé | fréquence :', SR)

In [ ]:
# Générer : 3 essais par audio, Whisper choisit le meilleur. Reprise automatique.
import csv, time
import numpy as np
import soundfile as sf

CHAMPS = ['fichier', 'phrase', 'score_1', 'score_2', 'score_3', 'duree_s', 'retenu', 'douteux']
GRAINES = [GRAINE, GRAINE + 1, GRAINE + 2]

def nettoyer(audio, sr, marge=0.05, cible=0.97):
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim > 1:
        audio = audio.mean(axis=0)
    if audio.size == 0:
        return audio
    seuil = max(1e-3, 0.02 * float(np.max(np.abs(audio))))
    forts = np.where(np.abs(audio) > seuil)[0]
    if forts.size:
        audio = audio[max(0, forts[0] - int(sr * marge)):min(len(audio), forts[-1] + int(sr * marge))]
    crete = float(np.max(np.abs(audio))) or 1.0
    return (audio * (cible / crete)).astype(np.float32)

def generer(phrase, graine):
    torch.manual_seed(graine); torch.cuda.manual_seed_all(graine)
    with torch.inference_mode():
        wav = oolel.generate(phrase, audio_prompt_path=REF, cfg_weight=0.5, exaggeration=0.2, temperature=0.3)
    wav = wav.detach().float().cpu().numpy() if hasattr(wav, 'cpu') else np.asarray(wav)
    return nettoyer(wav, SR)

lignes = []
if os.path.exists(RAPPORT):
    with open(RAPPORT, encoding='utf-8-sig') as f:
        lignes = list(csv.DictReader(f))
faits = {l['fichier'] for l in lignes}
a_faire = [p for p in PHRASES if not (p['fichier'] in faits and os.path.exists(f"{SORTIE}/{p['fichier']}"))]
print(len(a_faire), 'audios à générer')
debut = time.perf_counter()
for n, item in enumerate(a_faire, 1):
    cands = []
    for g in GRAINES:
        audio = generer(item['phrase'], g)
        duree = len(audio) / SR
        duree_ok, _ = controler_duree(duree, item['phrase'])
        try:
            score = similarite(item['phrase'], transcrire(audio, SR))
        except Exception:
            score = 0.0
        cands.append({'graine': g, 'audio': audio, 'duree': duree, 'score': score, 'duree_ok': duree_ok})
    i = choisir_meilleur_candidat(cands)
    r = cands[i]
    sf.write(f"{SORTIE}/{item['fichier']}", r['audio'], SR, subtype='PCM_16')
    lignes = [l for l in lignes if l['fichier'] != item['fichier']] + [{
        'fichier': item['fichier'], 'phrase': item['phrase'],
        'score_1': round(cands[0]['score'], 3), 'score_2': round(cands[1]['score'], 3), 'score_3': round(cands[2]['score'], 3),
        'duree_s': round(r['duree'], 2), 'retenu': f"{i + 1} (graine {r['graine']})",
        'douteux': 'oui' if est_douteux(r['score'], r['duree_ok']) else 'non'}]
    with open(RAPPORT, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=CHAMPS); w.writeheader(); w.writerows(lignes)
    reste = (time.perf_counter() - debut) / n * (len(a_faire) - n) / 60
    print(f"[{n}/{len(a_faire)}] {item['fichier']} score={r['score']:.2f} | reste ≈ {reste:.0f} min")
print('Terminé.')

In [ ]:
# Vérifier le dossier et préparer le zip pour installer_voix_oolel.py
manquants = [p['fichier'] for p in PHRASES if not os.path.exists(f"{SORTIE}/{p['fichier']}")]
assert not manquants, f"{len(manquants)} audios manquants : relance la cellule de génération."
shutil.copy(RAPPORT, f'{SORTIE}/rapport_ajouts_conversation.csv')
archive = shutil.make_archive('./sortie_voix/ajouts_conversation', 'zip', SORTIE)
print(
    '7 audios OK — sur ton PC : python scripts/installer_voix_oolel.py --conversation sortie_voix/ajouts_conversation.zip'
)
